# Wprowadzenie do PyTorcha – budowanie modeli z `nn.Sequential`

W dzisiejszych zajęciach rozpoczniemy pracę z **PyTorchem** – jedną z najpopularniejszych bibliotek do uczenia głębokiego. PyTorch udostępnia wygodny interfejs do pracy z tensorami, automatyczne obliczanie gradientów oraz elastyczne tworzenie sieci neuronowych.

W kolejnych zadaniach skupimy się na praktycznym zrozumieniu, **jak działają warstwy sieci w pełni połączonych (Fully Connected – FC)** oraz **warstwy konwolucyjne (Convolutional – CNN)**. 

Dzisiejszy materiał jest mocno inspirowany kursem [NYU Deep Learning](https://github.com/Atcold/NYU-DLSP20/tree/master), ale będzie uproszczony, aby najpierw zrozumieć podstawy.

---

## PyTorch w skrócie
PyTorch opiera się na dwóch głównych elementach:

### **1. Tensory (`torch.Tensor`)**
Struktury danych podobne do `numpy.array`, ale działające na CPU lub GPU (CUDA).
Umożliwiają:
- obliczenia matematyczne,
- przechowywanie gradientów,
- przechowywanie wag modeli.

### **2. Automatyczna różniczkowalność (`autograd`)**
PyTorch śledzi operacje na tensorach i automatycznie liczy gradienty podczas treningu.
Jeśli tensor ma `requires_grad=True`, to PyTorch będzie znał jego pochodną po funkcji straty.

---

## Budowanie modeli z `nn.Module` i `nn.Sequential`
Do tworzenia sieci neuronowych PyTorch udostępnia moduł `torch.nn`, który zawiera warstwy takie jak:
- `nn.Linear` – warstwa w pełni połączona,
- `nn.Conv2d` – warstwa konwolucyjna,
- `nn.ReLU`, `nn.Sigmoid`, `nn.Tanh` – funkcje aktywacji,
- `nn.Sequential` – szybkie budowanie modeli „od lewej do prawej”.

### Co to jest `nn.Sequential`?

`nn.Sequential` pozwala zdefiniować model warstwa po warstwie **bez pisania własnej klasy**.  
To najprostszy sposób, aby zrozumieć przepływ danych przez sieć.

Przykładowy model FC:

```python
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(784, 128),   # wejście: 784 cech, wyjście: 128 neuronów
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 10)      # 10 klas
)

# Przygotowanie środowiska programistycznego

In [1]:
import torch
import torch.nn as nn
from matplotlib.pyplot import plot, title, axis, figure, gca, gcf, rcParams
from numpy import clip
import urllib
from PIL import Image
from matplotlib import pyplot as plt
import numpy as np
import torch
from IPython.display import HTML, display, clear_output
from torch import nn, optim
from math import pi as π
rcParams['figure.max_open_warning'] = 100

Kilka pomocniczych funkcji do wizualizacji

In [3]:
def set_default(figsize=(10, 10), dpi=100):
    plt.style.use(['dark_background', 'bmh'])
    plt.rc('axes', facecolor='k')
    plt.rc('figure', facecolor='k')
    plt.rc('figure', figsize=figsize, dpi=dpi)


def plot_data(X, y, d=0, auto=False, zoom=1, title=None):
    
    X = X.cpu()
    y = y.cpu()
    plt.scatter(X.numpy()[:, 0], X.numpy()[:, 1], c=y, s=20, cmap=plt.cm.Spectral)
    plt.axis('square')
    plt.axis(np.array((-1.1, 1.1, -1.1, 1.1)) * zoom)
    if auto is True: plt.axis('equal')
    plt.axis('off')

    _m, _c = 0, '.15'
    plt.axvline(0, ymin=_m, color=_c, lw=1, zorder=0)
    plt.axhline(0, xmin=_m, color=_c, lw=1, zorder=0)
    if title is not None:
        plt.title(title)

def plot_model(X, y, model):
    model.cpu()
    mesh = np.arange(-1.1, 1.1, 0.01)
    xx, yy = np.meshgrid(mesh, mesh)
    with torch.no_grad():
        data = torch.from_numpy(np.vstack((xx.reshape(-1), yy.reshape(-1))).T).float()
        Z = model(data).detach()
    Z = np.argmax(Z, axis=1).reshape(xx.shape)
    plt.contourf(xx, yy, Z, cmap=plt.cm.Spectral, alpha=0.3)
    plot_data(X, y)

url = "https://www.fuw.edu.pl/~mpoziomska/data/ziegler.png"
zieger = Image.open(urllib.request.urlopen(url))
zieger = np.array(zieger) / 255.0

def show_scatterplot(X, colors, title='', axis=True):
    colors = zieger[colors[:,0], colors[:,1]]
    X = X.numpy()
    # plt.figure()
    plt.axis('equal')
    plt.scatter(X[:, 0], X[:, 1], c=colors, s=30)
    # plt.grid(True)
    plt.title(title)
    plt.axis('off')
    _m, _c = 0, '.15'
    if axis:
        plt.axvline(0, ymin=_m, color=_c, lw=1, zorder=0)
        plt.axhline(0, xmin=_m, color=_c, lw=1, zorder=0)


def plot_bases(bases, plotting=True, width=0.04):
    bases[2:] -= bases[:2]
    # if plot_bases.a: plot_bases.a.set_visible(False)
    # if plot_bases.b: plot_bases.b.set_visible(False)
    if plotting:
        plot_bases.a = plt.arrow(*bases[0], *bases[2], width=width, color='r', zorder=10, alpha=1., length_includes_head=True)
        plot_bases.b = plt.arrow(*bases[1], *bases[3], width=width, color='g', zorder=10, alpha=1., length_includes_head=True)


plot_bases.a = None
plot_bases.b = None


def show_mat(mat, vect, prod, threshold=-1):
    # Subplot grid definition
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, sharex=False, sharey=True,
                                        gridspec_kw={'width_ratios':[5,1,1]})
    # Plot matrices
    cax1 = ax1.matshow(mat.numpy(), clim=(-1, 1))
    ax2.matshow(vect.numpy(), clim=(-1, 1))
    cax3 = ax3.matshow(prod.numpy(), clim=(threshold, 1))

    # Set titles
    ax1.set_title(f'A: {mat.size(0)} \u00D7 {mat.size(1)}')
    ax2.set_title(f'a^(i): {vect.numel()}')
    ax3.set_title(f'p: {prod.numel()}')

    # Remove xticks for vectors
    ax2.set_xticks(tuple())
    ax3.set_xticks(tuple())

    # Plot colourbars
    fig.colorbar(cax1, ax=ax2)
    fig.colorbar(cax3, ax=ax3)

    # Fix y-axis limits
    ax1.set_ylim(bottom=max(len(prod), len(vect)) - 0.5)


colors = dict(
    aqua='#8dd3c7',
    yellow='#ffffb3',
    lavender='#bebada',
    red='#fb8072',
    blue='#80b1d3',
    orange='#fdb462',
    green='#b3de69',
    pink='#fccde5',
    grey='#d9d9d9',
    violet='#bc80bd',
    unk1='#ccebc5',
    unk2='#ffed6f',
)


def _cstr(s, color='black'):
    if s == ' ':
        return f'<text style=color:#000;padding-left:10px;background-color:{color}> </text>'
    else:
        return f'<text style=color:#000;background-color:{color}>{s} </text>'

# print html
def _print_color(t):
    display(HTML(''.join([_cstr(ti, color=ci) for ti, ci in t])))

# get appropriate color for value
def _get_clr(value):
    colors = ('#85c2e1', '#89c4e2', '#95cae5', '#99cce6', '#a1d0e8',
              '#b2d9ec', '#baddee', '#c2e1f0', '#eff7fb', '#f9e8e8',
              '#f9e8e8', '#f9d4d4', '#f9bdbd', '#f8a8a8', '#f68f8f',
              '#f47676', '#f45f5f', '#f34343', '#f33b3b', '#f42e2e')
    value = int((value * 100) / 5)
    if value == len(colors): value -= 1  # fixing bugs...
    return colors[value]

def _visualise_values(output_values, result_list):
    text_colours = []
    for i in range(len(output_values)):
        text = (result_list[i], _get_clr(output_values[i]))
        text_colours.append(text)
    _print_color(text_colours)

def print_colourbar():
    color_range = torch.linspace(-2.5, 2.5, 20)
    to_print = [(f'{x:.2f}', _get_clr((x+2.5)/5)) for x in color_range]
    _print_color(to_print)


# Let's only focus on the last time step for now
# First, the cell state (Long term memory)
def plot_state(data, state, b, decoder):
    actual_data = decoder(data[b, :, :].numpy())
    seq_len = len(actual_data)
    seq_len_w_pad = len(state)
    for s in range(state.size(2)):
        states = torch.sigmoid(state[:, b, s])
        _visualise_values(states[seq_len_w_pad - seq_len:], list(actual_data))


In [ ]:
# Set style (needs to be in a new cell)
%matplotlib inline
set_default()
torch.manual_seed(0)

## Wybór urządzenia do obliczeń (CPU vs GPU)

Trening sieci neuronowych może być wykonywany na dwóch typach urządzeń:

- **CPU** – procesor komputera  
- **GPU (CUDA)** – karta graficzna, która znacznie przyspiesza obliczenia równoległe

PyTorch umożliwia bardzo łatwe przenoszenie obliczeń pomiędzy tymi urządzeniami. 

In [5]:
device = "cpu" #torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

## Inicjalizacja danych

In [ ]:
# generate some points in 2-D space
n_points = 1_000
X = torch.randn(n_points, 2).to(device)
# colors [0 – 511]^2
x_min = -1.5 #X.min(0)[0] #+ 1
x_max = +1.5 #X.max(0)[0] #- 1
colors = (X - x_min) / (x_max - x_min)
colors =  (colors * 511).short().cpu().numpy()
colors = clip(colors, 0, 511)
figure().add_axes([0, 0, 1, 1])
show_scatterplot(X, colors, title='X')
OI = torch.cat((torch.zeros(2, 2), torch.eye(2))).to(device) # Wektory bazowe
plot_bases(OI)
plt.show()

# Jak działają warstwy liniowe w sieciach neuronowych?

W PyTorchu warstwa liniowa (`nn.Linear`) wykonuje **przekształcenie liniowe** wejściowego wektora:


$y = Wx + b$

gdzie:  
- $x$ — wektor wejściowy,  
- $W$ — macierz wag,  
- $b$ — wektor przesunięcia (bias),  
- $y$ — wektor wyjściowy.

Tego typu transformacje są fundamentem pełnych sieci połączeń (Fully Connected Networks, FC).  
Choć sama operacja wydaje się prosta, ma bardzo bogatą interpretację geometryczną.

---

## Przekształcenia liniowe jako operacje geometryczne

Macierz $W$ może **rozciągać**, **spłaszczać**, **obracać**, **odwracać** lub **przesuwać** (w połączeniu z biasem) przestrzeń wejściową.

Aby zrozumieć, jak to działa, możemy zdekomponować $W$ przy pomocy **dekompozycji SVD**:

$
\begin{equation}
    W = U
  \left[ {\begin{array}{cc}
   s_1 & 0 \\
   0 & s_2 \\
  \end{array} } \right]
  V^\top
\end{equation}
$
gdzie:

- $s_1$, $s_2$ — **wartości singularne**, określające skalowanie wzdłuż głównych osi przekształcenia,  
- $U$, $V$ — macierze ortogonalne, odpowiedzialne za **rotację** i **odbicia** punktów.

Następnie liczymy:

$y = Wx$

co oznacza, że każdy punkt $x$ jest przesuwany w nowe miejsce w przestrzeni.

---

## Co robią poszczególne elementy?

### 🔹 Wartości singularne $s_1$, $s_2$
- **Duże wartości** → przestrzeń jest **rozciągana** w danym kierunku  
- **Małe wartości** → przestrzeń jest **ściskana**, punkty są **bliżej siebie**

To wyjaśnia, dlaczego warstwy liniowe mogą „wydobywać” lub „ukrywać” różne kierunki informacji w danych.

### 🔹 Macierze $U$ i $V$
Są to macierze ortogonalne — działają jak:

- **rotacje**,  
- **odbicia**,  
- **zmiana bazy**,  

ale bez zmiany skali.

Możemy interpretować przekształcenie jako:

1. **Obróć** punkty macierzą $V$ 
2. **Rozciągnij/spłaszcz** przestrzeń przez wartości singularne  
3. **Ponownie obróć** przez macierz $U$

---

## Dlaczego to ważne dla sieci neuronowych?

Warstwa liniowa nie zmienia „kształtu” danych losowo — ona:

- wzmacnia pewne kierunki w danych (te, które są istotne dla klasyfikacji),
- tłumi inne kierunki (szum),
- reorganizuje przestrzeń tak, aby kolejne warstwy (lub funkcje aktywacji) mogły lepiej rozdzielić klasy.

W połączeniu z nieliniowościami (ReLU, tanh, sigmoid) pozwala to tworzyć złożone modele zdolne do odwzorowania skomplikowanych zależności.

---

## Zastosowanie w PyTorchu

W PyTorchu warstwa liniowa to:

```python
layer = nn.Linear(in_features=2, out_features=2)
```

# Wizualizacja przekształceń liniowych na przykładzie losowych macierzy

W tym fragmencie kodu przyjrzymy się, jak działają **przekształcenia liniowe** na dwuwymiarowych danych oraz jak można interpretować ich efekty przy pomocy **dekompozycji wartości osobliwych (SVD)**.


In [ ]:
figure()
show_scatterplot(X, colors, title='X')
plot_bases(OI)
plt.show()

for i in range(10):
    figure()
    # create a random matrix
    W = torch.randn(2, 2).to(device)
    # transform points
    Y = X @ W.t()
    # compute singular values
    U, S, V = torch.svd(W)
    # plot transformed points
    show_scatterplot(Y, colors, title='y = Wx, singular values : [{:.3f}, {:.3f}]'.format(S[0], S[1]))
    # transform the basis
    new_OI = OI @ W
    # plot old and new basis
    plot_bases(OI)
#     plot_bases(new_OI)
    plt.show()

# Transformacje liniowe w PyTorchu

W PyTorchu przekształcenia liniowe realizujemy przy pomocy warstwy **`nn.Linear`**.  

In [ ]:
# Inicjalizacja modelu
model = nn.Sequential(
        nn.Linear(2, 2, bias=False)
)
model.to(device) # przeniesienie modelu na wybrane urządzenie
with torch.no_grad(): # Wyłączenie obliczania gradientów - przydatne przy inferencji bez treningu
    Y = model(X) # Przepuszczenie danych przez model
    figure()
    show_scatterplot(Y, colors)
    plot_bases(model(OI))
    plt.show()

# Transformacja nieliniowa: odwzorowanie punktów na kwadrat

* Przekształcenia liniowe mogą **obracać**, **odbijać**, **rozciągać** i **ściskać**, ale **nie mogą wyginać** przestrzeni.  
* Do tego potrzebujemy **nieliniowości**.  
* Możemy (przybliżenie) odwzorować punkty na kwadrat, najpierw **rozciągając** je przez czynnik $s$, a następnie **ściskając** funkcją `tanh`:

$
   f(x)= \tanh \left(
  \left[ {\begin{array}{cc}
   s & 0 \\
   0 & s \\
  \end{array} } \right]  
  x
  \right)
$

Wizualizacja funkcji `tanh`

In [ ]:
z = torch.linspace(-10, 10, 101)
s = torch.tanh(z)
figure()
plot(z.numpy(), s.numpy())
title('tanh() non linearity');
plt.show()

W tej komórce rozszerzamy wcześniejsze przekształcenie liniowe o **nieliniowość** za pomocą funkcji aktywacji `tanh`.  

In [ ]:
figure()
show_scatterplot(X, colors, title='X')
plot_bases(OI)
plt.show()

model = nn.Sequential(
        nn.Linear(2, 2, bias=False),
        nn.Tanh()
)

model.to(device)

for s in range(1, 6):
    figure()
    W = s * torch.eye(2) # tworzy macierz diagonalną, która rozciąga punkty równomiernie w obu kierunkach
    model[0].weight.data.copy_(W) # ręcznie ustawia macierz wag warstwy liniowej
    Y = model(X).data # przepuszcza dane przez model
    show_scatterplot(Y, colors, title=f'f(x), s={s}')
    plot_bases(OI, width=0.01)
    plt.show()

# Zadanie

**Proszę:** Analogicznie zwizualizować i zbadać działanie aktywacji `ReLu` oraz `Sigmoid`.

**Wskazówka:** Zwizualizuj `ReLu` w taki sposób: `s = nn.ReLU()(z)`

In [ ]:
# Wizualizacja ReLU

#BEGIN_SOLUTION
...
#END_SOLUTION

In [ ]:
# Vizualizacja działania ReLU w sieci
#BEGIN_SOLUTION
...
#END_SOLUTION

In [ ]:
# Wizualizacja sigmoidy
#BEGIN_SOLUTION
...
#END_SOLUTION

In [ ]:
# Vizualizacja działania sigmoidy w sieci
#BEGIN_SOLUTION
...
#END_SOLUTION

Teraz zobaczmy jak zadziałają sieci z losowymi wagami.

**Proszę:** Zbadaj jak działają obie warstwy: `Tanh` i `ReLU`.

In [ ]:
# Najpierw prosta płytka sieć
figure()
show_scatterplot(X, colors, title='x')
plt.show()

n_hidden = 5

# NL = nn.ReLU()
NL = nn.Tanh()

models = list()

for i in range(5):
    # create 1-layer neural networks with random weights
    model = nn.Sequential(
            nn.Linear(2, n_hidden), 
            NL, 
            nn.Linear(n_hidden, 2)
        )
    model.to(device)
    models.append(model)
    with torch.no_grad():
        Y = model(X)
    figure()
    show_scatterplot(Y, colors, title='f(x)')
    # plot_bases(OI)
    plt.show()

Teraz głębsza sieć

In [ ]:
figure()
show_scatterplot(X, colors, title='x')
plt.show()

n_hidden = 5

# NL = nn.ReLU()
NL = nn.Tanh()

for i in range(5):
    model = nn.Sequential(
        nn.Linear(2, n_hidden), 
        NL, 
        nn.Linear(n_hidden, n_hidden), 
        NL, 
        nn.Linear(n_hidden, n_hidden), 
        NL, 
        nn.Linear(n_hidden, n_hidden), 
        NL, 
        nn.Linear(n_hidden, 2)
    )
    model.to(device)
    with torch.no_grad():
        Y = model(X).detach()
    figure()
    show_scatterplot(Y, colors, title='f(x)', axis=False)
    plt.show()

In [17]:
def interpolate(X_in, X_out, steps, p=1/50, plotting_grid=False, ratio='1:1'):
    N = 1000
    for t in range(steps):
        # a = (t / (steps - 1)) ** p
        a = ((p + 1)**(t / (steps - 1)) - 1) / p
        gca().cla()
#         plt.text(0, 5, action, color='w', horizontalalignment='center', verticalalignment='center')
        show_scatterplot(a * X_out + (1 - a) * X_in, colors, title='f(x)')

        if plotting_grid: plot_grid(a * X_out[N:] + (1 - a) * X_in[N:])
        gcf().canvas.draw()

# Generowanie danych do spirali

In [22]:
# Nowe funkcje pomocnicze

def set_default(figsize=(10, 10), dpi=100):
    plt.style.use(['dark_background', 'bmh'])
    plt.rc('axes', facecolor='k')
    plt.rc('figure', facecolor='k')
    plt.rc('figure', figsize=figsize, dpi=dpi)


def plot_data(X, y, d=0, auto=False, zoom=1, title='Training data (x, y)'):
    X = X.cpu()
    y = y.cpu()
    s = plt.scatter(X.numpy()[:, 0], X.numpy()[:, 1], c=y, s=20, cmap=plt.cm.Spectral)
    plt.axis('square')
    plt.axis(np.array((-1.1, 1.1, -1.1, 1.1)) * zoom)
    if auto is True:
        plt.axis('equal')
    plt.axis('off')

    _m, _c = 0, '.35'
    plt.axvline(0, ymin=_m, color=_c, lw=1)
    plt.axhline(0, xmin=_m, color=_c, lw=1)
    plt.title(title)
    return s



def plot_model(X, y, model):
    model.cpu()
    mesh = torch.arange(-1.1, 1.11, 0.01)
    xx, yy = torch.meshgrid(mesh, mesh, indexing='xy')
    with torch.no_grad():
        data = torch.stack((xx.reshape(-1), yy.reshape(-1)), dim=1)
        Z = model(data)
    Z = Z.argmax(dim=1).reshape(xx.shape)
    plt.contourf(xx.numpy(), yy.numpy(), Z, cmap=plt.cm.Spectral, alpha=0.3)
    plot_data(X, y)
    plt.title('Model decision boundaries')


def plot_embeddings(X, y, model, zoom=10):
    # Use forward hook to get internal embeddings of the second last layer
    layer_outputs = {}

    def get_layer_outputs(name):
        def hook(model, input, output):
            layer_outputs[name] = output

        return hook

    layer = model[-2]

    if layer.__class__ == torch.nn.modules.linear.Linear and layer.out_features == 2:
        layer.register_forward_hook(get_layer_outputs("low_dim_embeddings"))
        with torch.no_grad():
            model(X)  # pass data through model to populate layer_outputs
        plot_data(
            layer_outputs["low_dim_embeddings"],
            y,
            zoom=zoom,
            title="Low dim embeddings",
        )
        last_layer = model[-1]
        mesh = torch.arange(-1.1, 1.1, 0.01) * zoom
        xx, yy = torch.meshgrid(mesh, mesh, indexing="ij")
        with torch.no_grad():
            data = torch.stack((xx.reshape(-1), yy.reshape(-1)), dim=1)
            Z = last_layer(data)
        Z = Z.argmax(dim=1).reshape(xx.shape)
        plt.contourf(xx.numpy(), yy.numpy(), Z, cmap=plt.cm.Spectral, alpha=0.3, levels=y.max().item())
    else:
        print(
            "Cannot plot: second-last layer is not a linear layer"
            f" with output in R^2 (it is {layer})"
        )


def acc(l, y):
    score, predicted = torch.max(l, 1)
    return (y == predicted).sum().float() / len(y)


def overwrite(string):
    print(string)
    clear_output(wait=True)


def plot_2d_energy_levels(X, y, energy, v=None, l=None):
    xx, yy, F, k, K = energy
    if not v: vmin = vmax = None
    else: vmin, vmax = v
    if not l: levels = None
    else: levels = torch.arange(l[0], l[1], l[2])
    plt.figure(figsize=(12, 10))
    plt.pcolormesh(xx.numpy(), yy.numpy(), F, vmin=vmin, vmax=vmax)
    plt.colorbar()
    cnt = plt.contour(xx.numpy(), yy.numpy(), F, colors='w', linewidths=1, levels=levels)
    plt.clabel(cnt, inline=True, fontsize=10, colors='w')
    s = plot_data(X, y)
    plt.legend(*s.legend_elements(), title='Classes', loc='lower right')
    plt.axvline(color='0.55', lw=1)
    plt.axhline(color='0.55', lw=1)
    plt.axis([-1.5, 1.5, -1.5, 1.5])
    ȳ = torch.zeros(K).int(); ȳ[k] = 1
    plt.title(f'Free energy F(x, y = {ȳ.tolist()})')


def plot_3d_energy_levels(X, y, energy, v=None, l=None, cbl=None):
    xx, yy, F, k, K = energy
    if not v: vmin = vmax = None
    else: vmin, vmax = v
    if not l: levels = None
    else: levels = torch.arange(l[0], l[1], l[2])
    fig = plt.figure(figsize=(9.5, 6), facecolor='k')
    ax = fig.add_subplot(projection='3d')
    cnt = ax.contour(xx.numpy(), yy.numpy(), F, levels=levels, vmin=vmin, vmax=vmax)
    ax.scatter(X[:,0], X[:,1], zs=0, c=y, cmap=plt.cm.Spectral)
    ax.xaxis.set_pane_color(color=(0,0,0))
    ax.yaxis.set_pane_color(color=(0,0,0))
    ax.zaxis.set_pane_color(color=(0,0,0))

    vmin, vmax = cnt.get_clim()
    ax.set_zlim3d(vmin, vmax)
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    if not cbl: cbl = l
    else: cbl = torch.arange(cbl[0], cbl[1], cbl[2])
    sm = plt.cm.ScalarMappable(norm=norm, cmap=cnt.cmap)
    sm.set_array([])
    fig.colorbar(sm, ticks=cbl, ax=ax)
    ȳ = torch.zeros(K).int(); ȳ[k] = 1
    plt.title(f'Free energy F(x, y = {ȳ.tolist()})')
    plt.tight_layout()
    return fig, ax

In [24]:
seed = 12345
torch.manual_seed(seed)
N = 1000  # num_samples_per_class
n = 2     # input dimensions
K = 5     # num_classes
d = 100   # num_hidden_units

In [ ]:
# Generate spirals

t = torch.linspace(0, 1, N)
a = 0.8 * t + 0.2  # amplitude 0.2 → 1.0
X = list()
y = list()
for k in range(K):
    θ = (2 * t + k) * 2 * π / K + 0.2 * torch.randn(N)
    X.append(torch.stack((a * θ.sin(), a * θ.cos()), dim=1))
    y.append(torch.zeros(N, dtype=torch.long).fill_(k))
X = torch.cat(X)
y = torch.cat(y)

print("Shapes:")
print("X:", tuple(X.size()))
print("y:", tuple(y.size()))

In [ ]:
# And visualise them
plt.figure()
plot_data(X, y)
plt.show()

# Wprowadzenie do procesu uczenia modelu w PyTorchu

Uczenie modelu w PyTorchu odbywa się poprzez powtarzanie czterech podstawowych kroków:  
1. **Propagacja w przód (forward pass)**  
2. **Obliczenie funkcji straty**  
3. **Propagacja wstecz (backward pass)**  
4. **Aktualizacja parametrów**  

To jest podstawowy cykl uczenia każdej sieci neuronowej — od najprostszych modeli liniowych po duże sieci głębokie.

**Proszę:** Przy fragmencie do inicjalizacji modelu jest kilka opcji architektury: liniowa/nieliniowa, z przewężeniem lub bez. Sprawdź jak wygląda wynik w zależności od wyboru tych opcji. 

In [ ]:
learning_rate = 1e-3
lambda_l2 = 1e-5

model = nn.Sequential(
    nn.Linear(n, d),
    nn.ReLU(),  # Zobacz jaka jest różnica gdy ta linia jest zakomentowana
    nn.Linear(d, K)  # (Opcjonalnie) Jeżeli zakomentujesz tą linijkę i odkomentujesz tą poniższą, 
                        # to będziesz miał/a sieć z dwiema warstwami ukrytymi i przestrzenią przewężoną, którą niżej można zwizualizować
    # nn.Linear(d, 2), nn.Linear(2, K)
)
model.to(device)

# Cross entropy given the linear output
C = nn.CrossEntropyLoss(reduction='none') # standardowa funkcja straty do klasyfikacji wieloklasowej

# Using Adam optimiser
optimiser = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=lambda_l2) # popularny optymalizator, automatycznie dobiera kroki aktualizacji

# Full-batch training loop
for t in range(2_000):
    
    # Feed forward to get the linear sum s
    s = model(X)
    
    # Compute the free energy F and loss L
    F = C(s, y) # liczy funkcję straty dla każdego przykładu z osobna
    L = F.mean() # średnia funkcja straty po wszystkich przykładach
    
    # Zero the gradients
    optimiser.zero_grad() # zeruje gradienty z poprzedniego kroku
    
    # Backward pass to compute and accumulate the gradient
    # of the free energy with respect to our learnable params
    L.backward() # liczy gradienty funkcji straty względem wag sieci
    
    # Update params
    optimiser.step() # aktualizuje wagi sieci zgodnie z wyliczonymi gradientami
    
    # Display epoch, L, and accuracy
    overwrite(f'[EPOCH]: {t}, [LOSS]: {L.item():.6f}, [ACCURACY]: {acc(s, y):.3f}')

Jeżeli architektura modelu nie jest zbyt złożona, można go łatwo przedstawić w ten sposób

In [ ]:
print(model)

Teraz zwizualizujmy sobie wyniki naszej klasyfikacji

In [ ]:
figure()
plot_model(X, y, model)
plt.show()

(Opcjonalnie) Jeżeli masz odkomentowaną linijkę z przewężeniem

In [30]:
# figure()
# plot_embeddings(X, y, model, zoom=10)
# plt.show()